# Tutorial 06: Generating maps of the observed population

To perform inference on the observed data in the [ATNF Pulsar Catalogue](https://www.atnf.csiro.au/research/pulsar/psrcat/) combined with the radio fluxes from the TPA program on MeerKat [(Posselt et al., 2023)](https://ui.adsabs.harvard.edu/abs/2023MNRAS.520.4582P/abstract) with an optimized neural network, we need to convert the corresponding data into the same representation that is used to optimize our machine learning pipeline. That is, we need to produce the same kind of density maps as outlined above for the observed pulsar population.

To do this, we use the `mlpoppyns/generator/generate_observed_data.py` script. We can, for example, run:
```commandline
python mlpoppyns/generator/generate_observed_data.py --path_atnf data/observations/atnf_full_nobinary_24-09-2024_with_errors.csv --path_meerkat data/observations/meerkat_tpa_posselt_2023.csv --path_xray data/observations/thermal_NS_05-11-2024.csv --save_dir output/generator_observed --resolution_dyn_radio 32 --resolution_ppdot_radio 32 --resolution_dyn_xray 32 --resolution_ppdot_xray 32
```
This will read the files `atnf_full_nobinary_06-08-2024.csv`, `meerkat_tpa_posselt_2023.csv` and `thermal_NS_05-11-2024.csv` in the directory `data/observations` and generate the maps.

As for Tutorial 05, we can specify the map type (either `array`, `array_kde`, `image` or `image_kde`) with the argument `--data_type` and the resolution with the arguments `--resolution_dyn_radio`, `--resolution_ppdot_radio`, `--resolution_dyn_xray`, `--resolution_ppdot_xray` and whether filtering the detected X-ray population to consider only young magnetars and XDINSs with the flag `--filter_young_xdins`. 

With the above command, we then generate the following nine maps:

* four position density maps in ICRS coordinates: one for each of the three radio surveys and the X-ray survey modeled by the simulator.
* Four $P-\dot{P}$ density maps: one for each of the three simulated radio surveys and the X-ray survey.
* Four $P-\dot{P}$ density maps weighted by the logarithm of the flux: one for each of the three simulated radio surveys and the X-ray survey.

Moreover, the above command also produces a `dataset_observed.csv` file containing summary information of the different
maps. Note that the underlying ground truths are, of course, unknown here. The corresponding entries in the CSV file,
required for compatibility purposes, are therefore left empty.

In [ ]:
import argparse
import collections
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys

import utilities.plot_settings
from mlpoppyns.simulator.config_simulator import cfg
from mlpoppyns.generator.generate_observed_data import generate_dataset

WARNING: If you see a warning here, make sure to set the path to the repository in the simulator configuration file. Otherwise, the examples below will not run.

## Setting up and running the generator

Before running the generator script, we configure it by passing several arguments. This includes the type of representation for the maps (either `array` or `image`) and the resolutions for the dynamical and $P-\dot{P}$ maps.

We also specify the locations of the ATNF Pulsar Catalogue data and the TPA information as well as the output directory where the simulation results will be saved.

In [ ]:
atnf_dir = "../../data/observations/atnf_full_nobinary_24-09-2024_with_errors.csv"
meerkat_dir = "../../data/observations/meerkat_tpa_posselt_2023.csv"
xray_dir = "../../data/observations/thermal_NS_05-11-2024.csv"
output_dir = "output/generator_observed"

We run the generator by calling the `generate_dataset` function from the `mlpoppyns.generator.generate_observed_data` module for a specific choice of parameters.

In [ ]:
generator_args = argparse.Namespace(
    path_atnf=atnf_dir,
    path_meerkat=meerkat_dir,
    path_xray=xray_dir,
    save_dir=output_dir,
    data_type="array",
    resolution_dyn_radio=32,
    resolution_ppdot_radio=32,
    resolution_dyn_xray=32,
    resolution_ppdot_xray=32,
    filter_young_xdins=False
)

In [ ]:
generate_dataset(generator_args)

## Visualizing the generated maps

To display the maps, we first load the corresponding `.npy` arrays.

In [ ]:
ppdot_PMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_density_map_ppdot_0.npy")
)
ppdot_HTRU = np.load(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_density_map_ppdot_0.npy")
)
ppdot_SMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_density_map_ppdot_0.npy")
)
ppdot_xray = np.load(
    pathlib.Path().joinpath(output_dir, "survey_xray_density_map_ppdot_0.npy")
)
ppdot_flux_PMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_flux_map_ppdot_0.npy")
)
ppdot_flux_HTRU = np.load(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_flux_map_ppdot_0.npy")
)
ppdot_flux_SMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_flux_map_ppdot_0.npy")
)
ppdot_flux_xray = np.load(
    pathlib.Path().joinpath(output_dir, "survey_xray_flux_map_ppdot_0.npy")
)
position_PMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_density_map_radec_0.npy")
)
position_HTRU = np.load(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_density_map_radec_0.npy")
)
position_SMPS = np.load(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_density_map_radec_0.npy")
)
position_xray = np.load(
    pathlib.Path().joinpath(output_dir, "survey_xray_density_map_radec_0.npy")
)

Transposing the maps for correct visualization with imshow.

In [ ]:
ppdot_PMPS = ppdot_PMPS.T
ppdot_HTRU = ppdot_HTRU.T
ppdot_SMPS = ppdot_SMPS.T
ppdot_xray = ppdot_xray.T
ppdot_flux_PMPS = ppdot_flux_PMPS.T
ppdot_flux_HTRU = ppdot_flux_HTRU.T
ppdot_flux_SMPS = ppdot_flux_SMPS.T
ppdot_flux_xray = ppdot_flux_xray.T
position_PMPS = position_PMPS.T
position_SMPS = position_SMPS.T
position_HTRU = position_HTRU.T
position_xray = position_xray.T

Producing the $P-\dot{P}$ maps for each of the three surveys.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_PMPS, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"$P-\dot{P}$ density map: PMPS", fontsize=30, x=0.6)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_SMPS, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"$P-\dot{P}$ density map: SMPS", fontsize=30, x=0.6)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_HTRU, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"$P-\dot{P}$ density map: HTRU", fontsize=30, x=0.6)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_xray, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"$P-\dot{P}$ density map: X-ray", fontsize=30, x=0.6)

plt.show()

Producing the $P-\dot{P}$ maps weighted with the radio fluxes for each of the three surveys.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_flux_PMPS, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label(r"Average $\log_{10}(S_{\rm radio} \, [{\rm Jy}])$")
fig.suptitle(
    r"Fluxes displayed in $P-\dot{P}$ space: PMPS", fontsize=30, x=0.6
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_flux_SMPS, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label(r"Average $\log_{10}(S_{\rm radio} \, [{\rm Jy}])$")
fig.suptitle(
    r"Fluxes displayed in $P-\dot{P}$ space: SMPS", fontsize=30, x=0.6
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_flux_HTRU, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label(r"Average $\log_{10}(S_{\rm radio} \, [{\rm Jy}])$")
fig.suptitle(
    r"Fluxes displayed in $P-\dot{P}$ space: HTRU", fontsize=30, x=0.6
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(ppdot_flux_xray, cmap="viridis", origin="lower")
ax.set_xlabel("$P$ bin")
ax.set_ylabel("$\dot{P}$ bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label(r"Average $\log_{10}(S_{\rm X,abs} \, [{\rm erg} \, {\rm s}^{-1} \, {\rm cm}^{-2}])$")
fig.suptitle(
    r"Fluxes displayed in $P-\dot{P}$ space: X-ray", fontsize=30, x=0.6
)

plt.show()

Generating the sky position maps.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(position_PMPS, cmap="viridis", origin="lower")
ax.set_xlabel("RA bin")
ax.set_ylabel("DEC bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"Sky density map: PMPS", fontsize=30, x=0.5)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(position_SMPS, cmap="viridis", origin="lower")
ax.set_xlabel("RA bin")
ax.set_ylabel("DEC bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"Sky density map: SMPS", fontsize=30, x=0.5)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(position_HTRU, cmap="viridis", origin="lower")
ax.set_xlabel("RA bin")
ax.set_ylabel("DEC bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"Sky density map: HTRU", fontsize=30, x=0.5)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

map = ax.imshow(position_xray, cmap="viridis", origin="lower")
ax.set_xlabel("RA bin")
ax.set_ylabel("DEC bin")
colorbar = fig.colorbar(map, ax=ax)
colorbar.set_label("Number of NSs")
fig.suptitle(r"Sky density map: X-ray", fontsize=30, x=0.5)

plt.show()